In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/THESIS_DEEPFAKE"
folders = [
    "00_DOCUMENTATION",
    "01_DATASET_METADATA",
    "02_FACEForensics",
    "03_CELEB_DF",
    "04_DFDC",
    "05_PHILIPPINE",
    "06_PREPROCESSED",
    "07_MODELS",
    "08_EXPERIMENTS",
    "09_CHECKPOINTS",
    "10_RESULTS",
    "11_LOGS",
]
for folder in folders:
    os.makedirs(f"{base}/{folder}", exist_ok=True)

print("Created THESIS_DEEPFAKE structure on ellainesky.ellaine!")

Mounted at /content/drive
Created THESIS_DEEPFAKE structure on ellainesky.ellaine!


In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm scikit-learn

if not os.path.exists('/content/frames/clean'):
    !cp /content/drive/MyDrive/THESIS_DEEPFAKE/06_PREPROCESSED/extracted_frames.zip /content/
    !unzip -q /content/extracted_frames.zip -d /content/frames
    !rm /content/extracted_frames.zip

!ls /content/frames

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
clean  tier1_moderate  tier2_heavy


In [1]:
%%writefile /content/train_models.py
import io
import time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

THESIS_DIR = Path("/content/drive/MyDrive/THESIS_DEEPFAKE")
FRAMES_CLEAN_DIR = Path("/content/frames/clean")
SPLITS_DIR = THESIS_DIR / "01_DATASET_METADATA"
CKPT_DIR = THESIS_DIR / "09_CHECKPOINTS"
RESULTS_DIR = THESIS_DIR / "10_RESULTS"
LOGS_DIR = THESIS_DIR / "11_LOGS"

for d in (CKPT_DIR, RESULTS_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_CACHE = {}

MODEL_CONFIGS = {
    "efficientnet": {
        "img_size": 224,
        "batch_size": 32,
        "epochs": 30,
        "lr": 1e-4,
        "optimizer": "adam",
        "loss": "bce_logits",
        "is_sequence": False,
        "mean": (0.485, 0.456, 0.406),
        "std": (0.229, 0.224, 0.225),
    },
    "xception": {
        "img_size": 299,
        "batch_size": 16,
        "epochs": 30,
        "lr": 1e-4,
        "optimizer": "adam",
        "loss": "bce_logits",
        "is_sequence": False,
        "mean": (0.5, 0.5, 0.5),
        "std": (0.5, 0.5, 0.5),
    },
    "cnn_lstm": {
        "img_size": 224,
        "batch_size": 4,
        "epochs": 30,
        "lr": 1e-4,
        "optimizer": "adam",
        "loss": "bce_logits",
        "is_sequence": True,
        "mean": (0.485, 0.456, 0.406),
        "std": (0.229, 0.224, 0.225),
    },
    "genconvit": {
        "img_size": 224,
        "batch_size": 32,
        "epochs": 30,
        "lr": 1e-4,
        "optimizer": "adam",
        "loss": "ce",
        "is_sequence": False,
        "mean": (0.485, 0.456, 0.406),
        "std": (0.229, 0.224, 0.225),
    },
    "fakeformer": {
        "img_size": 112,
        "batch_size": 32,
        "epochs": 50,
        "lr": 5e-5,
        "optimizer": "adamw",
        "loss": "bce",
        "is_sequence": False,
        "mean": (0.485, 0.456, 0.406),
        "std": (0.229, 0.224, 0.225),
    },
}


def preload_dev_images(video_ids, root_dir: Path):
    missing = [v for v in video_ids if (v, 0) not in IMAGE_CACHE]
    if not missing:
        return
    print(f"Caching {len(missing) * 16} frames in RAM for fast training...")
    for vid_id in missing:
        vid_dir = root_dir / vid_id
        for k in range(16):
            IMAGE_CACHE[(vid_id, k)] = (vid_dir / f"frame_{k:02d}.jpg").read_bytes()
    total_mb = sum(len(b) for b in IMAGE_CACHE.values()) / (1024 * 1024)
    print(f"RAM cache ready ({total_mb:.1f} MB).")


def read_cached_frame(vid_id: str, k: int, root_dir: Path) -> Image.Image:
    raw = IMAGE_CACHE.get((vid_id, k))
    if raw is not None:
        return Image.open(io.BytesIO(raw)).convert("RGB")
    return Image.open(root_dir / vid_id / f"frame_{k:02d}.jpg").convert("RGB")


class CNNLSTMDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
        feat_dim = self.cnn.num_features
        self.lstm = nn.LSTM(input_size=feat_dim, hidden_size=256, num_layers=1, batch_first=True)
        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        b, t, c, h, w = x.shape
        feats = self.cnn(x.view(b * t, c, h, w)).view(b, t, -1)
        out, _ = self.lstm(feats)
        return self.fc(out[:, -1, :])


class GenConViTED(nn.Module):
    def __init__(self):
        super().__init__()
        self.convnext = timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        self.swin = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)
        fused_dim = self.convnext.num_features + self.swin.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Dropout(0.2),
            nn.Linear(fused_dim, 2),
        )

    def forward(self, x):
        f1 = self.convnext(x)
        f2 = self.swin(x)
        return self.head(torch.cat([f1, f2], dim=1))


def build_model(model_name: str) -> nn.Module:
    if model_name == "efficientnet":
        return timm.create_model("efficientnet_b0", pretrained=True, num_classes=1)
    if model_name == "xception":
        return timm.create_model("xception", pretrained=True, num_classes=1)
    if model_name == "cnn_lstm":
        return CNNLSTMDetector()
    if model_name == "genconvit":
        return GenConViTED()
    if model_name == "fakeformer":
        return timm.create_model("vit_small_patch8_224.dino", pretrained=True, img_size=112, num_classes=1)
    raise ValueError(f"Unknown model: {model_name}")


class FrameDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path, transform):
        self.samples = []
        self.root_dir = root_dir
        self.transform = transform
        for _, row in df.iterrows():
            vid_id = row["video_id"]
            label = int(row["label"])
            for k in range(16):
                self.samples.append((vid_id, k, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vid_id, k, label = self.samples[idx]
        img = read_cached_frame(vid_id, k, self.root_dir)
        return self.transform(img), label, vid_id


class SequenceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path, transform):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        vid_id = row["video_id"]
        label = int(row["label"])
        frames = [
            self.transform(read_cached_frame(vid_id, k, self.root_dir))
            for k in range(16)
        ]
        return torch.stack(frames, dim=0), label, vid_id


def train_model_cv(model_name: str, folds=(0, 1, 2, 3, 4)):
    cfg = MODEL_CONFIGS[model_name]
    print(f"\n==========================================")
    print(f"Training {model_name.upper()} | Config: {cfg}")
    print(f"==========================================")

    dev_df = pd.read_csv(SPLITS_DIR / "dev_720.csv")
    preload_dev_images(dev_df["video_id"].tolist(), FRAMES_CLEAN_DIR)

    train_tf = transforms.Compose([
        transforms.Resize((cfg["img_size"], cfg["img_size"])),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(cfg["mean"], cfg["std"]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((cfg["img_size"], cfg["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(cfg["mean"], cfg["std"]),
    ])

    summary_rows = []

    for fold in folds:
        ckpt_path = CKPT_DIR / f"{model_name}_fold{fold}_best.pth"
        log_path = LOGS_DIR / f"{model_name}_fold{fold}_history.csv"

        if ckpt_path.exists() and log_path.exists():
            print(f"[Fold {fold}] Already completed and saved on Drive. Skipping.")
            continue

        fold_df = pd.read_csv(SPLITS_DIR / f"fold_{fold}.csv")
        train_df = fold_df[fold_df["split"] == "train"].reset_index(drop=True)
        val_df = fold_df[fold_df["split"] == "val"].reset_index(drop=True)

        if cfg["is_sequence"]:
            train_ds = SequenceDataset(train_df, FRAMES_CLEAN_DIR, train_tf)
            val_ds = SequenceDataset(val_df, FRAMES_CLEAN_DIR, val_tf)
        else:
            train_ds = FrameDataset(train_df, FRAMES_CLEAN_DIR, train_tf)
            val_ds = FrameDataset(val_df, FRAMES_CLEAN_DIR, val_tf)

        train_loader = DataLoader(
            train_ds,
            batch_size=cfg["batch_size"],
            shuffle=True,
            num_workers=2,
            pin_memory=True,
            persistent_workers=True,
            prefetch_factor=2,
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=cfg["batch_size"],
            shuffle=False,
            num_workers=2,
            pin_memory=True,
            persistent_workers=True,
            prefetch_factor=2,
        )

        model = build_model(model_name).to(DEVICE)

        if cfg["optimizer"] == "adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
        else:
            optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])

        if cfg["loss"] == "bce_logits":
            criterion = nn.BCEWithLogitsLoss()
        elif cfg["loss"] == "ce":
            criterion = nn.CrossEntropyLoss()
        else:
            criterion = nn.BCELoss()

        scaler = torch.amp.GradScaler("cuda")
        best_auc = -1.0
        best_acc = 0.0
        history = []

        for epoch in range(1, cfg["epochs"] + 1):
            t0 = time.time()
            model.train()
            train_loss = 0.0

            for x, y, _ in train_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast("cuda"):
                    logits = model(x)

                logits_fp32 = logits.float()
                if cfg["loss"] == "bce_logits":
                    loss = criterion(logits_fp32.squeeze(-1), y.float())
                elif cfg["loss"] == "ce":
                    loss = criterion(logits_fp32, y.long())
                else:
                    probs = torch.clamp(torch.sigmoid(logits_fp32.squeeze(-1)), 1e-6, 1.0 - 1e-6)
                    loss = criterion(probs, y.float())

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item() * x.size(0)

            train_loss /= len(train_ds)

            model.eval()
            vid_probs = {}
            vid_labels = {}
            with torch.no_grad():
                for x, y, vid_ids in val_loader:
                    x = x.to(DEVICE, non_blocking=True)
                    with torch.amp.autocast("cuda"):
                        logits = model(x)
                        if cfg["loss"] == "ce":
                            probs = torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy()
                        else:
                            probs = torch.sigmoid(logits.squeeze(-1).float()).cpu().numpy()

                    for vid_id, p, lbl in zip(vid_ids, probs, y.numpy()):
                        vid_probs.setdefault(vid_id, []).append(float(p))
                        vid_labels[vid_id] = int(lbl)

            y_true = [vid_labels[v] for v in vid_probs]
            y_score = [np.mean(vid_probs[v]) for v in vid_probs]
            y_pred = [1 if s >= 0.5 else 0 for s in y_score]

            val_acc = accuracy_score(y_true, y_pred)
            val_auc = roc_auc_score(y_true, y_score)
            val_f1 = f1_score(y_true, y_pred)
            elapsed = time.time() - t0

            history.append({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_acc": val_acc,
                "val_auc": val_auc,
                "val_f1": val_f1,
            })

            if val_auc > best_auc:
                best_auc = val_auc
                best_acc = val_acc
                torch.save(model.state_dict(), ckpt_path)

            print(
                f"[Fold {fold} | Epoch {epoch:02d}/{cfg['epochs']}] "
                f"Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | "
                f"Best AUC: {best_auc:.4f} ({elapsed:.1f}s)"
            )

        pd.DataFrame(history).to_csv(log_path, index=False)
        summary_rows.append({"model": model_name, "fold": fold, "best_val_acc": best_acc, "best_val_auc": best_auc})

    if summary_rows:
        sum_df = pd.DataFrame(summary_rows)
        sum_df.to_csv(RESULTS_DIR / f"{model_name}_cv_summary.csv", index=False)
        print(f"\n[{model_name.upper()} 5-Fold Summary]\n", sum_df)

Writing /content/train_models.py


In [ ]:
from train_models import train_model_cv
train_model_cv("efficientnet")


Training EFFICIENTNET | Config: {'img_size': 224, 'batch_size': 32, 'epochs': 30, 'lr': 0.0001, 'optimizer': 'adam', 'loss': 'bce_logits', 'is_sequence': False, 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225)}


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

[Fold 0 | Epoch 01/30] Loss: 0.7587 | Val Acc: 0.7361 | Val AUC: 0.8200 | Best AUC: 0.8200 (59.8s)
[Fold 0 | Epoch 02/30] Loss: 0.1628 | Val Acc: 0.7917 | Val AUC: 0.8574 | Best AUC: 0.8574 (51.5s)
[Fold 0 | Epoch 03/30] Loss: 0.1023 | Val Acc: 0.7778 | Val AUC: 0.8519 | Best AUC: 0.8574 (50.7s)
[Fold 0 | Epoch 04/30] Loss: 0.0696 | Val Acc: 0.8125 | Val AUC: 0.8426 | Best AUC: 0.8574 (52.6s)
[Fold 0 | Epoch 05/30] Loss: 0.0668 | Val Acc: 0.7986 | Val AUC: 0.8441 | Best AUC: 0.8574 (49.8s)
[Fold 0 | Epoch 06/30] Loss: 0.0509 | Val Acc: 0.8056 | Val AUC: 0.8465 | Best AUC: 0.8574 (52.0s)
[Fold 0 | Epoch 07/30] Loss: 0.0494 | Val Acc: 0.7708 | Val AUC: 0.8295 | Best AUC: 0.8574 (50.3s)
[Fold 0 | Epoch 08/30] Loss: 0.0447 | Val Acc: 0.8542 | Val AUC: 0.8748 | Best AUC: 0.8748 (50.2s)
[Fold 0 | Epoch 09/30] Loss: 0.0305 | Val Acc: 0.8194 | Val AUC: 0.8555 | Best AUC: 0.8748 (51.9s)
[Fold 0 | Epoch 10/30] Loss: 0.0343 | Val Acc: 0.8472 | Val AUC: 0.8920 | Best AUC: 0.8920 (49.9s)
[Fold 0 | 

In [ ]:
import importlib
from pathlib import Path
import train_models

p = Path("/content/train_models.py")
code = p.read_text()

old_block = """                with torch.amp.autocast("cuda"):
                    logits = model(x)
                    if cfg["loss"] == "bce_logits":
                        loss = criterion(logits.squeeze(-1), y.float())
                    elif cfg["loss"] == "ce":
                        loss = criterion(logits, y.long())
                    else:
                        probs = torch.sigmoid(logits.squeeze(-1).float())
                        loss = criterion(probs, y.float())"""

new_block = """                with torch.amp.autocast("cuda"):
                    logits = model(x)

                logits_fp32 = logits.float()
                if cfg["loss"] == "bce_logits":
                    loss = criterion(logits_fp32.squeeze(-1), y.float())
                elif cfg["loss"] == "ce":
                    loss = criterion(logits_fp32, y.long())
                else:
                    probs = torch.clamp(torch.sigmoid(logits_fp32.squeeze(-1)), 1e-6, 1.0 - 1e-6)
                    loss = criterion(probs, y.float())"""

if old_block in code:
    p.write_text(code.replace(old_block, new_block, 1))

importlib.reload(train_models)
train_models.train_model_cv("fakeformer")


Training FAKEFORMER | Config: {'img_size': 112, 'batch_size': 32, 'epochs': 50, 'lr': 5e-05, 'optimizer': 'adamw', 'loss': 'bce', 'is_sequence': False, 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225)}
Caching 11520 frames in RAM for fast training...
RAM cache ready (216.7 MB).
[Fold 0 | Epoch 01/50] Loss: 0.3879 | Val Acc: 0.7708 | Val AUC: 0.8374 | Best AUC: 0.8374 (35.9s)
[Fold 0 | Epoch 02/50] Loss: 0.1243 | Val Acc: 0.7431 | Val AUC: 0.8304 | Best AUC: 0.8374 (33.5s)
[Fold 0 | Epoch 03/50] Loss: 0.0835 | Val Acc: 0.7917 | Val AUC: 0.8582 | Best AUC: 0.8582 (33.1s)
[Fold 0 | Epoch 04/50] Loss: 0.0726 | Val Acc: 0.7431 | Val AUC: 0.8333 | Best AUC: 0.8582 (34.1s)
[Fold 0 | Epoch 05/50] Loss: 0.0586 | Val Acc: 0.7639 | Val AUC: 0.8565 | Best AUC: 0.8582 (35.7s)
[Fold 0 | Epoch 06/50] Loss: 0.0482 | Val Acc: 0.7639 | Val AUC: 0.8465 | Best AUC: 0.8582 (33.4s)
[Fold 0 | Epoch 07/50] Loss: 0.0519 | Val Acc: 0.7917 | Val AUC: 0.8601 | Best AUC: 0.8601 (33.8s)
[Fold 0 | Epoch 

In [ ]:
import train_models
train_models.train_model_cv("genconvit")


Training GENCONVIT | Config: {'img_size': 224, 'batch_size': 32, 'epochs': 30, 'lr': 0.0001, 'optimizer': 'adam', 'loss': 'ce', 'is_sequence': False, 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225)}
Caching 11520 frames in RAM for fast training...
RAM cache ready (216.7 MB).


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

[Fold 0 | Epoch 01/30] Loss: 0.5374 | Val Acc: 0.6736 | Val AUC: 0.7980 | Best AUC: 0.7980 (125.3s)
[Fold 0 | Epoch 02/30] Loss: 0.2154 | Val Acc: 0.7083 | Val AUC: 0.7928 | Best AUC: 0.7980 (122.5s)
[Fold 0 | Epoch 03/30] Loss: 0.1535 | Val Acc: 0.7639 | Val AUC: 0.8302 | Best AUC: 0.8302 (123.0s)
[Fold 0 | Epoch 04/30] Loss: 0.0896 | Val Acc: 0.7431 | Val AUC: 0.8181 | Best AUC: 0.8302 (121.7s)
[Fold 0 | Epoch 05/30] Loss: 0.0849 | Val Acc: 0.7917 | Val AUC: 0.8397 | Best AUC: 0.8397 (122.3s)
[Fold 0 | Epoch 06/30] Loss: 0.0620 | Val Acc: 0.8125 | Val AUC: 0.8470 | Best AUC: 0.8470 (121.0s)
[Fold 0 | Epoch 07/30] Loss: 0.0585 | Val Acc: 0.7847 | Val AUC: 0.8532 | Best AUC: 0.8532 (122.7s)
[Fold 0 | Epoch 08/30] Loss: 0.0534 | Val Acc: 0.7639 | Val AUC: 0.8708 | Best AUC: 0.8708 (121.8s)
[Fold 0 | Epoch 09/30] Loss: 0.0496 | Val Acc: 0.7639 | Val AUC: 0.8503 | Best AUC: 0.8708 (122.8s)
[Fold 0 | Epoch 10/30] Loss: 0.0490 | Val Acc: 0.7917 | Val AUC: 0.8594 | Best AUC: 0.8708 (121.8s)


In [3]:
!ls /content/drive/MyDrive/THESIS_DEEPFAKE/09_CHECKPOINTS/*.pth | wc -l

25


In [3]:
%%writefile /content/evaluate_models.py
import os
import gc
import time
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from scipy.stats import chi2, binomtest
import matplotlib.pyplot as plt

import train_models

THESIS_DIR = Path("/content/drive/MyDrive/THESIS_DEEPFAKE")
FRAMES_ROOT = Path("/content/frames")
SPLITS_DIR = THESIS_DIR / "01_DATASET_METADATA"
CKPT_DIR = THESIS_DIR / "09_CHECKPOINTS"
RESULTS_DIR = THESIS_DIR / "10_RESULTS"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS = list(train_models.MODEL_CONFIGS.keys())
TIERS = ["clean", "tier1_moderate", "tier2_heavy"]


def get_display_name(model_key: str) -> str:
    k = model_key.lower()
    if "efficientnet" in k:
        return "EfficientNet-B0"
    if "fakeformer" in k:
        return "FakeFormer-S"
    if "genconvit" in k:
        return "GenConViT-ED"
    if "cnn" in k or "lstm" in k:
        return "CNN-LSTM"
    if "xception" in k:
        return "Xception"
    return model_key


def resolve_ckpt_path(model_key: str, fold: int) -> Path:
    exact = CKPT_DIR / f"{model_key}_fold{fold}_best.pth"
    if exact.exists():
        return exact
    prefix = model_key.split("_")[0]
    matches = sorted(CKPT_DIR.glob(f"*{prefix}*fold{fold}*.pth"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"No checkpoint found for model='{model_key}', fold={fold} in {CKPT_DIR}")


def instantiate_model(model_key: str):
    for fn_name in ("build_model", "create_model", "get_model", "init_model"):
        if hasattr(train_models, fn_name):
            return getattr(train_models, fn_name)(model_key)
    for attr_name in dir(train_models):
        obj = getattr(train_models, attr_name)
        if isinstance(obj, type) and issubclass(obj, nn.Module) and obj is not nn.Module:
            try:
                return obj(model_key)
            except TypeError:
                continue
    raise AttributeError(
        f"Could not auto-instantiate model from train_models.py. Available attributes: {dir(train_models)}"
    )


def load_checkpoint(model, ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(ckpt, dict):
        state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    else:
        state = ckpt
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model


def resolve_test_df():
    df = pd.read_csv(SPLITS_DIR / "test_180.csv")
    vid_col = next(c for c in ("video_id", "id", "clip_id", "filename", "folder") if c in df.columns)
    lbl_col = next(c for c in ("label", "target", "y", "class") if c in df.columns)
    df = df.copy()
    df["vid_key"] = df[vid_col].astype(str).apply(lambda s: Path(s).stem if "." in Path(s).name else Path(s).name)
    df["y_true"] = df[lbl_col].astype(int)
    return df


def load_video_tensor(vid_key: str, tier: str, tfm):
    vid_dir = FRAMES_ROOT / tier / vid_key
    frame_paths = sorted(vid_dir.glob("*.jpg"))
    if len(frame_paths) == 0:
        raise FileNotFoundError(f"No frames found in {vid_dir}")
    imgs = [tfm(Image.open(p).convert("RGB")) for p in frame_paths]
    return torch.stack(imgs, dim=0)  # [T, 3, H, W]


def cochrans_q_test(correct_matrix: np.ndarray):
    n, k = correct_matrix.shape
    g = correct_matrix.sum(axis=0)
    l = correct_matrix.sum(axis=1)
    g_mean = g.mean()
    denom = k * l.sum() - np.sum(l ** 2)
    if denom == 0:
        return 0.0, k - 1, 1.0
    q_stat = (k * (k - 1) * np.sum((g - g_mean) ** 2)) / denom
    p_val = float(chi2.sf(q_stat, df=k - 1))
    return float(q_stat), k - 1, p_val


def mcnemar_pairwise_holm(correct_dict: dict):
    rows = []
    for m1, m2 in itertools.combinations(MODELS, 2):
        c1 = correct_dict[m1]
        c2 = correct_dict[m2]
        b = int(np.sum((c1 == 1) & (c2 == 0)))
        c = int(np.sum((c1 == 0) & (c2 == 1)))
        n_disc = b + c
        if n_disc > 0:
            chi2_cc = ((abs(b - c) - 1) ** 2) / n_disc
            p_chi2 = float(chi2.sf(chi2_cc, df=1))
            p_exact = float(binomtest(min(b, c), n=n_disc, p=0.5, alternative="two-sided").pvalue)
        else:
            chi2_cc, p_chi2, p_exact = 0.0, 1.0, 1.0

        rows.append({
            "model_a": get_display_name(m1),
            "model_b": get_display_name(m2),
            "a_right_b_wrong (b)": b,
            "a_wrong_b_right (c)": c,
            "discordant_pairs": n_disc,
            "mcnemar_chi2_cc": round(chi2_cc, 4),
            "p_value_chi2": p_chi2,
            "p_value_exact": p_exact,
        })

    res_df = pd.DataFrame(rows).sort_values("p_value_exact").reset_index(drop=True)
    m = len(res_df)
    adj_p = []
    prev = 0.0
    for idx, p in enumerate(res_df["p_value_exact"]):
        val = min(1.0, max(prev, (m - idx) * p))
        adj_p.append(val)
        prev = val
    res_df["p_value_holm"] = adj_p
    res_df["significant_0.05"] = res_df["p_value_holm"] < 0.05
    return res_df


def run_full_evaluation():
    test_df = resolve_test_df()
    y_true = test_df["y_true"].values
    vid_keys = test_df["vid_key"].tolist()
    print(f"Detected MODEL_CONFIGS keys: {MODELS}")
    print(f"Loaded test_180.csv: {len(test_df)} videos (Real={np.sum(y_true==0)}, Fake={np.sum(y_true==1)})")

    per_fold_records = []
    video_prob_records = []
    ensemble_probs = {tier: {} for tier in TIERS}

    for model_name in MODELS:
        disp_name = get_display_name(model_name)
        cfg = train_models.MODEL_CONFIGS[model_name]
        img_size = cfg["img_size"]
        mean = tuple(cfg["mean"])
        std = tuple(cfg["std"])
        is_seq = cfg["is_sequence"]

        tfm = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ])

        print(f"\n==================== Evaluating {disp_name} ({model_name}) ====================")
        for tier in TIERS:
            # Load only one tier at a time and release RAM immediately afterward
            vid_tensors = [load_video_tensor(vk, tier, tfm) for vk in vid_keys]
            fold_probs_list = []

            for fold in range(5):
                ckpt_path = resolve_ckpt_path(model_name, fold)
                model = instantiate_model(model_name)
                model = load_checkpoint(model, ckpt_path)

                with torch.no_grad():
                    dummy = vid_tensors[0].unsqueeze(0).to(DEVICE) if is_seq else vid_tensors[0].to(DEVICE)
                    _ = model(dummy)
                    if DEVICE.type == "cuda":
                        torch.cuda.synchronize()

                probs = []
                latencies_ms = []

                with torch.no_grad():
                    for vt in vid_tensors:
                        if DEVICE.type == "cuda":
                            torch.cuda.synchronize()
                        t0 = time.perf_counter()

                        inp = vt.unsqueeze(0).to(DEVICE) if is_seq else vt.to(DEVICE)
                        out = model(inp)
                        if isinstance(out, (tuple, list)):
                            out = out[0]
                        p = torch.sigmoid(out).mean().item()

                        if DEVICE.type == "cuda":
                            torch.cuda.synchronize()
                        t1 = time.perf_counter()

                        probs.append(p)
                        latencies_ms.append((t1 - t0) * 1000.0)

                probs = np.array(probs)
                preds = (probs >= 0.5).astype(int)
                fold_probs_list.append(probs)

                per_fold_records.append({
                    "model": model_name,
                    "display_name": disp_name,
                    "tier": tier,
                    "fold": fold,
                    "accuracy": accuracy_score(y_true, preds),
                    "precision": precision_score(y_true, preds, zero_division=0),
                    "recall": recall_score(y_true, preds, zero_division=0),
                    "f1": f1_score(y_true, preds, zero_division=0),
                    "roc_auc": roc_auc_score(y_true, probs),
                    "latency_ms_per_video": float(np.mean(latencies_ms)),
                })

                del model
                gc.collect()
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()

            # Free tier tensors from System RAM before loading the next tier
            del vid_tensors
            gc.collect()

            ens_p = np.mean(np.stack(fold_probs_list, axis=0), axis=0)
            ensemble_probs[tier][model_name] = ens_p

            for i, vk in enumerate(vid_keys):
                video_prob_records.append({
                    "tier": tier,
                    "video_id": vk,
                    "y_true": int(y_true[i]),
                    "model": model_name,
                    "display_name": disp_name,
                    "ensemble_prob": float(ens_p[i]),
                    "ensemble_pred": int(ens_p[i] >= 0.5),
                    "correct": int((ens_p[i] >= 0.5) == y_true[i]),
                })

            fold_df_sub = pd.DataFrame(per_fold_records)
            sub = fold_df_sub[(fold_df_sub["model"] == model_name) & (fold_df_sub["tier"] == tier)]
            print(
                f"  [{tier:<14}] 5-Fold Mean Acc: {sub['accuracy'].mean():.4f} ± {sub['accuracy'].std():.4f} | "
                f"Mean AUC: {sub['roc_auc'].mean():.4f} ± {sub['roc_auc'].std():.4f} | "
                f"Latency: {sub['latency_ms_per_video'].mean():.2f} ms/vid"
            )

        # Save incremental progress after each model completes
        pd.DataFrame(per_fold_records).to_csv(RESULTS_DIR / "test180_per_fold_metrics.csv", index=False)

    per_fold_df = pd.DataFrame(per_fold_records)
    per_fold_df.to_csv(RESULTS_DIR / "test180_per_fold_metrics.csv", index=False)

    vid_pred_df = pd.DataFrame(video_prob_records)
    vid_pred_df.to_csv(RESULTS_DIR / "test180_video_predictions.csv", index=False)

    summary_rows = []
    for model_name in MODELS:
        disp_name = get_display_name(model_name)
        clean_sub = per_fold_df[(per_fold_df["model"] == model_name) & (per_fold_df["tier"] == "clean")]
        clean_acc_mean = clean_sub["accuracy"].mean()
        clean_auc_mean = clean_sub["roc_auc"].mean()

        for tier in TIERS:
            sub = per_fold_df[(per_fold_df["model"] == model_name) & (per_fold_df["tier"] == tier)]
            ens_p = ensemble_probs[tier][model_name]
            ens_pred = (ens_p >= 0.5).astype(int)

            summary_rows.append({
                "model": disp_name,
                "tier": tier,
                "fold_acc_mean": round(sub["accuracy"].mean(), 4),
                "fold_acc_sd": round(sub["accuracy"].std(), 4),
                "fold_prec_mean": round(sub["precision"].mean(), 4),
                "fold_rec_mean": round(sub["recall"].mean(), 4),
                "fold_f1_mean": round(sub["f1"].mean(), 4),
                "fold_auc_mean": round(sub["roc_auc"].mean(), 4),
                "fold_auc_sd": round(sub["roc_auc"].std(), 4),
                "delta_acc_vs_clean": round(sub["accuracy"].mean() - clean_acc_mean, 4),
                "delta_auc_vs_clean": round(sub["roc_auc"].mean() - clean_auc_mean, 4),
                "ensemble_acc": round(accuracy_score(y_true, ens_pred), 4),
                "ensemble_f1": round(f1_score(y_true, ens_pred, zero_division=0), 4),
                "ensemble_auc": round(roc_auc_score(y_true, ens_p), 4),
                "latency_ms_per_video": round(sub["latency_ms_per_video"].mean(), 2),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(RESULTS_DIR / "test180_summary_table.csv", index=False)

    cochran_rows = []
    mcnemar_all = []

    for tier in TIERS:
        correct_dict = {
            m: ((ensemble_probs[tier][m] >= 0.5).astype(int) == y_true).astype(int)
            for m in MODELS
        }
        correct_mat = np.column_stack([correct_dict[m] for m in MODELS])
        q_stat, df_q, p_q = cochrans_q_test(correct_mat)
        cochran_rows.append({
            "tier": tier,
            "n_videos": len(y_true),
            "k_models": len(MODELS),
            "cochrans_Q": round(q_stat, 4),
            "df": df_q,
            "p_value": p_q,
            "significant_0.05": p_q < 0.05,
        })

        mc_df = mcnemar_pairwise_holm(correct_dict)
        mc_df.insert(0, "tier", tier)
        mcnemar_all.append(mc_df)

    cochran_df = pd.DataFrame(cochran_rows)
    mcnemar_df = pd.concat(mcnemar_all, ignore_index=True)

    cochran_df.to_csv(RESULTS_DIR / "stats_cochrans_q.csv", index=False)
    mcnemar_df.to_csv(RESULTS_DIR / "stats_mcnemar_pairwise.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), dpi=300)
    tier_titles = {
        "clean": "Clean (Uncompressed)",
        "tier1_moderate": "Tier 1: Moderate Degradation",
        "tier2_heavy": "Tier 2: Heavy Degradation",
    }
    for ax, tier in zip(axes, TIERS):
        for m in MODELS:
            ens_p = ensemble_probs[tier][m]
            fpr, tpr, _ = roc_curve(y_true, ens_p)
            auc_val = roc_auc_score(y_true, ens_p)
            ax.plot(fpr, tpr, lw=2, label=f"{get_display_name(m)} (AUC={auc_val:.3f})")
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
        ax.set_title(tier_titles[tier], fontsize=12, fontweight="bold")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right", fontsize=8.5)
        ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig_roc_curves_test180.png", dpi=300)
    plt.close()

    fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=300)
    x = np.arange(len(TIERS))
    for m in MODELS:
        disp_name = get_display_name(m)
        sub = summary_df[summary_df["model"] == disp_name]
        means = [sub[sub["tier"] == t]["fold_auc_mean"].values[0] for t in TIERS]
        sds = [sub[sub["tier"] == t]["fold_auc_sd"].values[0] for t in TIERS]
        ax.errorbar(x, means, yerr=sds, marker="o", capsize=4, lw=2, label=disp_name)

    ax.set_xticks(x)
    ax.set_xticklabels(["Clean", "Tier 1 (Moderate)", "Tier 2 (Heavy)"])
    ax.set_ylabel("5-Fold Mean ROC-AUC")
    ax.set_title("Detection Robustness Across Degradation Tiers (test_180)", fontweight="bold")
    ax.set_ylim(0.5, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig_degradation_robustness.png", dpi=300)
    plt.close()

    print("\n================ SUMMARY TABLE (test_180) ================")
    print(summary_df.to_string(index=False))
    print("\n================ COCHRAN'S Q TEST ================")
    print(cochran_df.to_string(index=False))
    print("\n================ PAIRWISE McNEMAR (HOLM-BONFERRONI) ================")
    print(mcnemar_df.to_string(index=False))
    print(f"\nAll CSV tables and 300-DPI figures saved to: {RESULTS_DIR}")

Overwriting /content/evaluate_models.py


In [4]:
import importlib
import evaluate_models

importlib.reload(evaluate_models)
evaluate_models.run_full_evaluation()

Detected MODEL_CONFIGS keys: ['efficientnet', 'xception', 'cnn_lstm', 'genconvit', 'fakeformer']
Loaded test_180.csv: 180 videos (Real=90, Fake=90)

==================== Evaluating EfficientNet-B0 (efficientnet) ====================


  [clean         ] 5-Fold Mean Acc: 0.7856 ± 0.0301 | Mean AUC: 0.8491 ± 0.0284 | Latency: 22.16 ms/vid
  [tier1_moderate] 5-Fold Mean Acc: 0.7267 ± 0.0262 | Mean AUC: 0.7694 ± 0.0371 | Latency: 20.29 ms/vid
  [tier2_heavy   ] 5-Fold Mean Acc: 0.5811 ± 0.0224 | Mean AUC: 0.6053 ± 0.0118 | Latency: 20.57 ms/vid

==================== Evaluating Xception (xception) ====================


/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


  [clean         ] 5-Fold Mean Acc: 0.8900 ± 0.0164 | Mean AUC: 0.9593 ± 0.0112 | Latency: 127.00 ms/vid


/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


  [tier1_moderate] 5-Fold Mean Acc: 0.7811 ± 0.0134 | Mean AUC: 0.8793 ± 0.0171 | Latency: 126.86 ms/vid


... (6 outputs left out)

  [clean         ] 5-Fold Mean Acc: 0.5111 ± 0.0286 | Mean AUC: 0.4985 ± 0.1294 | Latency: 142.82 ms/vid
  [tier1_moderate] 5-Fold Mean Acc: 0.5100 ± 0.0194 | Mean AUC: 0.5160 ± 0.0739 | Latency: 141.71 ms/vid
  [tier2_heavy   ] 5-Fold Mean Acc: 0.4989 ± 0.0025 | Mean AUC: 0.5407 ± 0.0414 | Latency: 141.54 ms/vid

==================== Evaluating FakeFormer-S (fakeformer) ====================


model.safetensors: reconstructing file:   0%|          |  0.00B / 86.7MB            

model.safetensors: downloading bytes:           |  0.00B            

  [clean         ] 5-Fold Mean Acc: 0.7411 ± 0.0306 | Mean AUC: 0.8069 ± 0.0333 | Latency: 51.94 ms/vid
  [tier1_moderate] 5-Fold Mean Acc: 0.7211 ± 0.0310 | Mean AUC: 0.7814 ± 0.0329 | Latency: 51.83 ms/vid
  [tier2_heavy   ] 5-Fold Mean Acc: 0.6456 ± 0.0341 | Mean AUC: 0.7122 ± 0.0492 | Latency: 51.52 ms/vid

================ SUMMARY TABLE (test_180) ================
          model           tier  fold_acc_mean  fold_acc_sd  fold_prec_mean  fold_rec_mean  fold_f1_mean  fold_auc_mean  fold_auc_sd  delta_acc_vs_clean  delta_auc_vs_clean  ensemble_acc  ensemble_f1  ensemble_auc  latency_ms_per_video
EfficientNet-B0          clean         0.7856       0.0301          0.7922         0.7756        0.7822         0.8491       0.0284              0.0000              0.0000        0.8000       0.7955        0.8889                 22.16
EfficientNet-B0 tier1_moderate         0.7267       0.0262          0.7491         0.6889        0.7132         0.7694       0.0371             -0.0589       

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm
!rm -rf /content/ph_data
!unzip -q /content/drive/MyDrive/THESIS_DEEPFAKE/06_PREPROCESSED/ph_dataset.zip -d /content/ph_data
!find /content/ph_data -maxdepth 3

Mounted at /content/drive
/content/ph_data
/content/ph_data/ph_test.csv
/content/ph_data/ph_in_the_wild
/content/ph_data/ph_in_the_wild/PH_FAKE_006
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_12.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_11.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_01.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_10.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_07.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_04.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_03.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_00.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_15.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_05.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_14.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_06.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_09.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_08.jpg
/content/ph_data/ph_in_the_wild/PH_FAKE_006/frame_02.j

In [3]:
%%writefile /content/evaluate_ph.py
import gc
import time
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from scipy.stats import chi2, binomtest
import matplotlib.pyplot as plt

import train_models

THESIS_DIR = Path("/content/drive/MyDrive/THESIS_DEEPFAKE")
PH_UNZIP_ROOT = Path("/content/ph_data")
CKPT_DIR = THESIS_DIR / "09_CHECKPOINTS"
RESULTS_DIR = THESIS_DIR / "10_RESULTS"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODELS = list(train_models.MODEL_CONFIGS.keys())


def get_display_name(model_key: str) -> str:
    k = model_key.lower()
    if "efficientnet" in k:
        return "EfficientNet-B0"
    if "fakeformer" in k:
        return "FakeFormer-S"
    if "genconvit" in k:
        return "GenConViT-ED"
    if "cnn" in k or "lstm" in k:
        return "CNN-LSTM"
    if "xception" in k:
        return "Xception"
    return model_key


def resolve_ckpt_path(model_key: str, fold: int) -> Path:
    exact = CKPT_DIR / f"{model_key}_fold{fold}_best.pth"
    if exact.exists():
        return exact
    prefix = model_key.split("_")[0]
    matches = sorted(CKPT_DIR.glob(f"*{prefix}*fold{fold}*.pth"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"No checkpoint found for model='{model_key}', fold={fold} in {CKPT_DIR}")


def instantiate_model(model_key: str):
    for fn_name in ("build_model", "create_model", "get_model", "init_model"):
        if hasattr(train_models, fn_name):
            return getattr(train_models, fn_name)(model_key)
    for attr_name in dir(train_models):
        obj = getattr(train_models, attr_name)
        if isinstance(obj, type) and issubclass(obj, nn.Module) and obj is not nn.Module:
            try:
                return obj(model_key)
            except TypeError:
                continue
    raise AttributeError("Could not auto-instantiate model from train_models.py")


def load_checkpoint(model, ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(ckpt, dict):
        state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    else:
        state = ckpt
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model


def cochrans_q_test(correct_matrix: np.ndarray):
    n, k = correct_matrix.shape
    g = correct_matrix.sum(axis=0)
    l = correct_matrix.sum(axis=1)
    g_mean = g.mean()
    denom = k * l.sum() - np.sum(l ** 2)
    if denom == 0:
        return 0.0, k - 1, 1.0
    q_stat = (k * (k - 1) * np.sum((g - g_mean) ** 2)) / denom
    p_val = float(chi2.sf(q_stat, df=k - 1))
    return float(q_stat), k - 1, p_val


def mcnemar_pairwise_holm(correct_dict: dict):
    rows = []
    for m1, m2 in itertools.combinations(MODELS, 2):
        c1 = correct_dict[m1]
        c2 = correct_dict[m2]
        b = int(np.sum((c1 == 1) & (c2 == 0)))
        c = int(np.sum((c1 == 0) & (c2 == 1)))
        n_disc = b + c
        if n_disc > 0:
            chi2_cc = ((abs(b - c) - 1) ** 2) / n_disc
            p_chi2 = float(chi2.sf(chi2_cc, df=1))
            p_exact = float(binomtest(min(b, c), n=n_disc, p=0.5, alternative="two-sided").pvalue)
        else:
            chi2_cc, p_chi2, p_exact = 0.0, 1.0, 1.0

        rows.append({
            "model_a": get_display_name(m1),
            "model_b": get_display_name(m2),
            "a_right_b_wrong (b)": b,
            "a_wrong_b_right (c)": c,
            "discordant_pairs": n_disc,
            "mcnemar_chi2_cc": round(chi2_cc, 4),
            "p_value_chi2": p_chi2,
            "p_value_exact": p_exact,
        })

    res_df = pd.DataFrame(rows).sort_values("p_value_exact").reset_index(drop=True)
    m = len(res_df)
    adj_p = []
    prev = 0.0
    for idx, p in enumerate(res_df["p_value_exact"]):
        val = min(1.0, max(prev, (m - idx) * p))
        adj_p.append(val)
        prev = val
    res_df["p_value_holm"] = adj_p
    res_df["significant_0.05"] = res_df["p_value_holm"] < 0.05
    return res_df


def run_ph_evaluation():
    csv_path = next(PH_UNZIP_ROOT.rglob("ph_test.csv"))
    frames_dir = next(PH_UNZIP_ROOT.rglob("ph_in_the_wild"))
    ph_df = pd.read_csv(csv_path)

    vid_keys = ph_df["video_id"].tolist()
    y_true = ph_df["label"].astype(int).values
    print(f"Device: {DEVICE} | Loaded ph_test.csv: {len(ph_df)} videos (Real={np.sum(y_true==0)}, Fake={np.sum(y_true==1)})")

    clean_baselines = {}
    t180_path = RESULTS_DIR / "test180_summary_table.csv"
    if t180_path.exists():
        t180_df = pd.read_csv(t180_path)
        for _, row in t180_df[t180_df["tier"] == "clean"].iterrows():
            clean_baselines[row["model"]] = (row["fold_acc_mean"], row["fold_auc_mean"])

    per_fold_records = []
    video_prob_records = []
    ensemble_probs = {}

    for model_name in MODELS:
        disp_name = get_display_name(model_name)
        cfg = train_models.MODEL_CONFIGS[model_name]
        img_size = cfg["img_size"]
        mean = tuple(cfg["mean"])
        std = tuple(cfg["std"])
        is_seq = cfg["is_sequence"]

        tfm = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ])

        vid_tensors = []
        for vk in vid_keys:
            fpaths = sorted((frames_dir / vk).glob("*.jpg"))
            imgs = [tfm(Image.open(p).convert("RGB")) for p in fpaths]
            vid_tensors.append(torch.stack(imgs, dim=0))

        fold_probs_list = []
        for fold in range(5):
            ckpt_path = resolve_ckpt_path(model_name, fold)
            model = load_checkpoint(instantiate_model(model_name), ckpt_path)

            probs = []
            with torch.no_grad():
                for vt in vid_tensors:
                    inp = vt.unsqueeze(0).to(DEVICE) if is_seq else vt.to(DEVICE)
                    out = model(inp)
                    if isinstance(out, (tuple, list)):
                        out = out[0]
                    probs.append(torch.sigmoid(out).mean().item())

            probs = np.array(probs)
            preds = (probs >= 0.5).astype(int)
            fold_probs_list.append(probs)

            per_fold_records.append({
                "model": model_name,
                "display_name": disp_name,
                "fold": fold,
                "accuracy": accuracy_score(y_true, preds),
                "precision": precision_score(y_true, preds, zero_division=0),
                "recall": recall_score(y_true, preds, zero_division=0),
                "f1": f1_score(y_true, preds, zero_division=0),
                "roc_auc": roc_auc_score(y_true, probs),
            })

            del model
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        del vid_tensors
        gc.collect()

        ens_p = np.mean(np.stack(fold_probs_list, axis=0), axis=0)
        ensemble_probs[model_name] = ens_p

        for i, row in ph_df.iterrows():
            video_prob_records.append({
                "video_id": row["video_id"],
                "pair_id": row["pair_id"],
                "platform": row["platform"],
                "y_true": int(y_true[i]),
                "model": model_name,
                "display_name": disp_name,
                "ensemble_prob": float(ens_p[i]),
                "ensemble_pred": int(ens_p[i] >= 0.5),
                "correct": int((ens_p[i] >= 0.5) == y_true[i]),
            })

    per_fold_df = pd.DataFrame(per_fold_records)
    per_fold_df.to_csv(RESULTS_DIR / "ph_per_fold_metrics.csv", index=False)

    vid_pred_df = pd.DataFrame(video_prob_records)
    vid_pred_df.to_csv(RESULTS_DIR / "ph_video_predictions.csv", index=False)

    summary_rows = []
    for model_name in MODELS:
        disp_name = get_display_name(model_name)
        sub = per_fold_df[per_fold_df["model"] == model_name]
        ens_p = ensemble_probs[model_name]
        ens_pred = (ens_p >= 0.5).astype(int)

        # Pairwise ranking accuracy: how often Fake prob > Real prob for the same person
        vsub = vid_pred_df[vid_pred_df["model"] == model_name]
        pair_wins = 0
        for pid, grp in vsub.groupby("pair_id"):
            p_fake = grp[grp["y_true"] == 1]["ensemble_prob"].values[0]
            p_real = grp[grp["y_true"] == 0]["ensemble_prob"].values[0]
            if p_fake > p_real:
                pair_wins += 1

        acc_mean = sub["accuracy"].mean()
        auc_mean = sub["roc_auc"].mean()
        c_acc, c_auc = clean_baselines.get(disp_name, (np.nan, np.nan))

        summary_rows.append({
            "model": disp_name,
            "ph_fold_acc_mean": round(acc_mean, 4),
            "ph_fold_acc_sd": round(sub["accuracy"].std(), 4),
            "ph_fold_prec_mean": round(sub["precision"].mean(), 4),
            "ph_fold_rec_mean": round(sub["recall"].mean(), 4),
            "ph_fold_f1_mean": round(sub["f1"].mean(), 4),
            "ph_fold_auc_mean": round(auc_mean, 4),
            "ph_fold_auc_sd": round(sub["roc_auc"].std(), 4),
            "domain_gap_acc_vs_clean": round(acc_mean - c_acc, 4) if not np.isnan(c_acc) else np.nan,
            "domain_gap_auc_vs_clean": round(auc_mean - c_auc, 4) if not np.isnan(c_auc) else np.nan,
            "ph_ensemble_acc": round(accuracy_score(y_true, ens_pred), 4),
            "ph_ensemble_f1": round(f1_score(y_true, ens_pred, zero_division=0), 4),
            "ph_ensemble_auc": round(roc_auc_score(y_true, ens_p), 4),
            "matched_pairs_ranked_correctly": f"{pair_wins}/10",
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(RESULTS_DIR / "ph_summary_table.csv", index=False)

    correct_dict = {
        m: ((ensemble_probs[m] >= 0.5).astype(int) == y_true).astype(int)
        for m in MODELS
    }
    correct_mat = np.column_stack([correct_dict[m] for m in MODELS])
    q_stat, df_q, p_q = cochrans_q_test(correct_mat)
    cochran_df = pd.DataFrame([{
        "dataset": "ph_in_the_wild",
        "n_videos": len(y_true),
        "k_models": len(MODELS),
        "cochrans_Q": round(q_stat, 4),
        "df": df_q,
        "p_value": p_q,
        "significant_0.05": p_q < 0.05,
    }])
    cochran_df.to_csv(RESULTS_DIR / "stats_ph_cochrans_q.csv", index=False)

    mcnemar_df = mcnemar_pairwise_holm(correct_dict)
    mcnemar_df.to_csv(RESULTS_DIR / "stats_ph_mcnemar_pairwise.csv", index=False)

    # Save 300-DPI ROC Curve for PH In-the-Wild Dataset
    fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=300)
    for m in MODELS:
        ens_p = ensemble_probs[m]
        fpr, tpr, _ = roc_curve(y_true, ens_p)
        auc_val = roc_auc_score(y_true, ens_p)
        ax.plot(fpr, tpr, lw=2, label=f"{get_display_name(m)} (AUC={auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
    ax.set_title("Philippine In-the-Wild Generalization ROC Curves (N=20)", fontweight="bold")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "fig_roc_curves_ph.png", dpi=300)
    plt.close()

    print("\n================ PH IN-THE-WILD SUMMARY TABLE ================")
    print(summary_df.to_string(index=False))
    print("\n================ PH COCHRAN'S Q TEST ================")
    print(cochran_df.to_string(index=False))
    print("\n================ PH PAIRWISE McNEMAR (HOLM-BONFERRONI) ================")
    print(mcnemar_df.to_string(index=False))
    print(f"\nAll PH evaluation results saved to: {RESULTS_DIR}")

Writing /content/evaluate_ph.py


In [4]:
import evaluate_ph
evaluate_ph.run_ph_evaluation()

Device: cuda | Loaded ph_test.csv: 20 videos (Real=10, Fake=10)


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth


/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 86.7MB            

model.safetensors: downloading bytes:           |  0.00B            


================ PH IN-THE-WILD SUMMARY TABLE ================
          model  ph_fold_acc_mean  ph_fold_acc_sd  ph_fold_prec_mean  ph_fold_rec_mean  ph_fold_f1_mean  ph_fold_auc_mean  ph_fold_auc_sd  domain_gap_acc_vs_clean  domain_gap_auc_vs_clean  ph_ensemble_acc  ph_ensemble_f1  ph_ensemble_auc matched_pairs_ranked_correctly
EfficientNet-B0              0.66          0.0652             0.8700              0.40           0.5342             0.654          0.0841                  -0.1256                  -0.1951             0.65          0.4615             0.74                           8/10
       Xception              0.63          0.0274             0.8929              0.32           0.4546             0.684          0.0777                  -0.2600                  -0.2753             0.65          0.4615             0.66                           8/10
       CNN-LSTM              0.63          0.0570             0.7462              0.42           0.5233             0.624        

In [2]:
%%writefile /content/run_sop3_sop4.py
import gc
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare
import torch

import train_models

THESIS_DIR = Path("/content/drive/MyDrive/THESIS_DEEPFAKE")
CKPT_DIR = THESIS_DIR / "09_CHECKPOINTS"
RESULTS_DIR = THESIS_DIR / "10_RESULTS"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS = list(train_models.MODEL_CONFIGS.keys())


def get_display_name(model_key: str) -> str:
    k = model_key.lower()
    if "efficientnet" in k:
        return "EfficientNet-B0"
    if "fakeformer" in k:
        return "FakeFormer-S"
    if "genconvit" in k:
        return "GenConViT-ED"
    if "cnn" in k or "lstm" in k:
        return "CNN-LSTM"
    if "xception" in k:
        return "Xception"
    return model_key


def resolve_ckpt_path(model_key: str, fold: int = 0) -> Path:
    exact = CKPT_DIR / f"{model_key}_fold{fold}_best.pth"
    if exact.exists():
        return exact
    prefix = model_key.split("_")[0]
    matches = sorted(CKPT_DIR.glob(f"*{prefix}*fold{fold}*.pth"))
    return matches[0]


def instantiate_model(model_key: str):
    for fn_name in ("build_model", "create_model", "get_model", "init_model"):
        if hasattr(train_models, fn_name):
            return getattr(train_models, fn_name)(model_key)
    raise AttributeError("Model builder not found in train_models.py")


def generate_complete_sop_tables():
    # 1. Load existing SOP 1 & SOP 2 tables
    t180_sum = pd.read_csv(RESULTS_DIR / "test180_summary_table.csv")
    t180_fold = pd.read_csv(RESULTS_DIR / "test180_per_fold_metrics.csv")
    ph_sum = pd.read_csv(RESULTS_DIR / "ph_summary_table.csv")
    ph_fold = pd.read_csv(RESULTS_DIR / "ph_per_fold_metrics.csv")
    cochran_180 = pd.read_csv(RESULTS_DIR / "stats_cochrans_q.csv")
    mcnemar_180 = pd.read_csv(RESULTS_DIR / "stats_mcnemar_pairwise.csv")
    cochran_ph = pd.read_csv(RESULTS_DIR / "stats_ph_cochrans_q.csv")

    # 2. SOP 3: Friedman omnibus significance tests across all 5 metrics
    metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
    friedman_rows = []

    for tier in ["clean", "tier1_moderate", "tier2_heavy"]:
        sub = t180_fold[t180_fold["tier"] == tier]
        for met in metrics:
            arrays = [
                sub[sub["model"] == m].sort_values("fold")[met].values
                for m in MODELS
            ]
            stat, pval = friedmanchisquare(*arrays)
            friedman_rows.append({
                "dataset_condition": f"test180_{tier}",
                "metric": met,
                "friedman_chi2": round(float(stat), 4),
                "df": len(MODELS) - 1,
                "p_value": float(pval),
                "significant_0.05": bool(pval < 0.05),
            })

    for met in metrics:
        arrays = [
            ph_fold[ph_fold["model"] == m].sort_values("fold")[met].values
            for m in MODELS
        ]
        stat, pval = friedmanchisquare(*arrays)
        friedman_rows.append({
            "dataset_condition": "ph_in_the_wild",
            "metric": met,
            "friedman_chi2": round(float(stat), 4),
            "df": len(MODELS) - 1,
            "p_value": float(pval),
            "significant_0.05": bool(pval < 0.05),
        })

    friedman_df = pd.DataFrame(friedman_rows)
    friedman_df.to_csv(RESULTS_DIR / "stats_friedman_all_metrics.csv", index=False)

    # 3. SOP 4: Computational Efficiency Profiling (Inference Time, Model Size, Params, RAM/GPU)
    lat_map = (
        t180_sum[t180_sum["tier"] == "clean"]
        .set_index("model")["latency_ms_per_video"]
        .to_dict()
    )
    clean_auc_map = (
        t180_sum[t180_sum["tier"] == "clean"]
        .set_index("model")["fold_auc_mean"]
        .to_dict()
    )
    ph_auc_map = ph_sum.set_index("model")["ph_fold_auc_mean"].to_dict()

    eff_rows = []
    for m in MODELS:
        disp = get_display_name(m)
        cfg = train_models.MODEL_CONFIGS[m]
        img_size = cfg["img_size"]
        is_seq = cfg["is_sequence"]

        ckpt_path = resolve_ckpt_path(m, fold=0)
        size_mb = ckpt_path.stat().st_size / (1024 ** 2)

        model = instantiate_model(m)
        ckpt = torch.load(ckpt_path, map_location="cpu")
        state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt)) if isinstance(ckpt, dict) else ckpt
        model.load_state_dict(state)

        total_params = sum(p.numel() for p in model.parameters())
        param_ram_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)

        model.to(DEVICE)
        model.eval()

        dummy = (
            torch.randn(1, 16, 3, img_size, img_size, device=DEVICE)
            if is_seq
            else torch.randn(16, 3, img_size, img_size, device=DEVICE)
        )

        # Warmup
        with torch.no_grad():
            _ = model(dummy)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()

        # Measure peak GPU memory & forward latency
        times_ms = []
        with torch.no_grad():
            for _ in range(10):
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()
                t0 = time.perf_counter()
                _ = model(dummy)
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()
                times_ms.append((time.perf_counter() - t0) * 1000.0)

        peak_vram_mb = (
            torch.cuda.max_memory_allocated() / (1024 ** 2)
            if DEVICE.type == "cuda"
            else 0.0
        )

        eff_rows.append({
            "model": disp,
            "input_resolution": f"{img_size}x{img_size}",
            "inference_time_ms_video": lat_map.get(disp, round(float(np.mean(times_ms)), 2)),
            "model_size_mb": round(size_mb, 2),
            "parameters_millions": round(total_params / 1e6, 2),
            "total_parameters_exact": int(total_params),
            "model_weights_ram_mb": round(param_ram_mb, 2),
            "peak_gpu_vram_mb": round(peak_vram_mb, 2),
            "clean_auc": clean_auc_map.get(disp, np.nan),
            "ph_auc": ph_auc_map.get(disp, np.nan),
        })

        del model, dummy
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    eff_df = pd.DataFrame(eff_rows)
    eff_df.to_csv(RESULTS_DIR / "sop4_computational_efficiency.csv", index=False)

    print("================ SOP 1 & 2.1: TEST_180 SUMMARY ================")
    print(t180_sum.to_string(index=False))
    print("\n================ SOP 2.2: PH IN-THE-WILD SUMMARY ================")
    print(ph_sum.to_string(index=False))
    print("\n================ SOP 3A: COCHRAN'S Q (ACCURACY) ================")
    print(pd.concat([cochran_180, cochran_ph], ignore_index=True).to_string(index=False))
    print("\n================ SOP 3B: FRIEDMAN TEST (ALL 5 METRICS) ================")
    print(friedman_df.to_string(index=False))
    print("\n================ SOP 3C: SIGNIFICANT PAIRWISE McNEMAR (p_holm < 0.05) ================")
    print(mcnemar_180[mcnemar_180["significant_0.05"] == True].to_string(index=False))
    print("\n================ SOP 4: COMPUTATIONAL EFFICIENCY ================")
    print(eff_df.to_string(index=False))

Writing /content/run_sop3_sop4.py


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import run_sop3_sop4
run_sop3_sop4.generate_complete_sop_tables()

Mounted at /content/drive


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/timm/models/_factory.py:230: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 86.7MB            

model.safetensors: downloading bytes:           |  0.00B            

================ SOP 1 & 2.1: TEST_180 SUMMARY ================
          model           tier  fold_acc_mean  fold_acc_sd  fold_prec_mean  fold_rec_mean  fold_f1_mean  fold_auc_mean  fold_auc_sd  delta_acc_vs_clean  delta_auc_vs_clean  ensemble_acc  ensemble_f1  ensemble_auc  latency_ms_per_video
EfficientNet-B0          clean         0.7856       0.0301          0.7922         0.7756        0.7822         0.8491       0.0284              0.0000              0.0000        0.8000       0.7955        0.8889                 22.16
EfficientNet-B0 tier1_moderate         0.7267       0.0262          0.7491         0.6889        0.7132         0.7694       0.0371             -0.0589             -0.0797        0.7667       0.7529        0.8137                 20.29
EfficientNet-B0    tier2_heavy         0.5811       0.0224          0.6136         0.4844        0.5273         0.6053       0.0118             -0.2044             -0.2438        0.5944       0.5166        0.6246                 20